# Converting PCI-DSS v4.0.1 PDF to JSON/Markdown

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ethanolivertroy/pcidss-docs-mcp/blob/main/tutorials/convert-pdf-to-json.ipynb)

---

## ⚠️ LEGAL NOTICE - READ CAREFULLY

### This Notebook Teaches You to Build an Internal Tool

**What this is:**
- ✅ A tutorial showing how to convert PCI-DSS PDFs to JSON for **YOUR INTERNAL USE**
- ✅ Educational content on building compliance tools
- ✅ Open source software (MIT licensed)

**What this is NOT:**
- ❌ A way to distribute or share PCI-DSS content
- ❌ A redistribution of copyrighted material
- ❌ A replacement for licensing PDFs from PCI SSC

### Your Responsibilities

By using this notebook, you agree to:
1. Download PCI-DSS PDFs from https://www.pcisecuritystandards.org/document_library/
2. Accept the PCI DSS License Agreement (click "ACCEPT" on their website)
3. Use converted data for **YOUR INTERNAL PURPOSES ONLY**
4. **NOT** share, redistribute, or publish converted JSON/Markdown
5. **NOT** commit converted data to public repositories

**PCI DSS License Key Terms:**
- "for internal purposes only" - You can use it, not share it
- "Licensee shall not sublicense" - You can't give it to others
- Users must accept license directly from PCI SSC

### Legal Safety

This approach is legally safe because:
- We teach you **HOW** to build a tool (legal)
- We don't distribute PCI DSS content (would be illegal)
- You license content directly from PCI SSC (their process)
- You use it internally only (compliant with license)

**Think of it like teaching someone to use a photocopier vs. distributing photocopies.**

**Questions?** See `../data/pdfs/README.md` for detailed legal explanation.

---

## Prerequisites

Before running this notebook, you need:

1. **PCI-DSS v4.0.1 PDF** - Downloaded from https://www.pcisecuritystandards.org/document_library/
2. **Python 3.9+** - Already available in Colab or local Jupyter
3. **pip** - Package installer (included with Python)

### Running Options

**Option A: Google Colab** (No setup required)
- Click the "Open in Colab" badge above
- Upload your PCI-DSS PDF when prompted
- Run all cells

**Option B: Local Jupyter**
```bash
pip install jupyter
jupyter notebook
```

## Step 1: Install Dependencies

We'll install:
- **docling** (>= 1.16.1) - PDF parsing and extraction library
- **PyMuPDF** (>= 1.24.0) - PDF processing backend

In [ ]:
!pip install -q docling>=1.16.1 PyMuPDF>=1.24.0

print("✅ Dependencies installed successfully!")

## Step 2: Upload PDF (Google Colab) or Verify PDF (Local)

### For Google Colab Users:

Run the cell below to upload your PCI-DSS PDF.

### For Local Jupyter Users:

Make sure you've placed the PDF at: `../data/pdfs/PCI-DSS-v4_0_1.pdf`

In [ ]:
import os
from pathlib import Path

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print("🌐 Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Running locally")

if IN_COLAB:
    from google.colab import files
    print("\n📤 Please upload your PCI-DSS v4.0.1 PDF...")
    uploaded = files.upload()
    
    # Get the uploaded filename
    pdf_filename = list(uploaded.keys())[0]
    pdf_path = Path(pdf_filename)
    
    print(f"\n✅ Uploaded: {pdf_path.name} ({pdf_path.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    # Local path
    pdf_path = Path("../data/pdfs/PCI-DSS-v4_0_1.pdf")
    
    if not pdf_path.exists():
        print(f"❌ PDF not found at: {pdf_path}")
        print("\nPlease:")
        print("1. Download PCI-DSS v4.0.1 from https://www.pcisecuritystandards.org/document_library/")
        print("2. Accept the license agreement")
        print(f"3. Place the PDF at: {pdf_path.absolute()}")
        raise FileNotFoundError(f"PDF not found: {pdf_path}")
    else:
        print(f"✅ Found PDF: {pdf_path} ({pdf_path.stat().st_size / 1024 / 1024:.1f} MB)")

## Step 3: Set Up Output Directory

We'll create a directory for the converted output.

In [ ]:
if IN_COLAB:
    output_dir = Path("converted")
else:
    output_dir = Path("../data/converted")

output_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Output directory ready: {output_dir.absolute()}")

## Part 2: Understanding PDF Structure with Docling

Before running the full conversion, let's explore how Docling extracts data from PDFs.

### Learning Objectives:
1. Inspect PDF structure (tables, headings, text)
2. Understand the **hybrid extraction** approach
3. Extract level 1 & 2 requirements from markdown
4. Extract level 3+ requirements from tables
5. Merge and validate the complete dataset

This hands-on section will teach you the techniques used in the conversion script.

In [ ]:
# Cell 2.1: Understanding PDF Structure

# Import required modules
import sys
import re
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

def create_configured_converter():
    """Configure Docling for text-layer PDFs (OCR disabled for speed)"""
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = False  # PCI-DSS has text layer - skip OCR for 2x speed
    pipeline_options.do_table_structure = True  # Extract table structure

    converter = DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    return converter

# Convert the PDF
print("🔄 Converting PDF with Docling (this may take 3-4 minutes)...")
print("   OCR disabled: 2x faster for text-layer PDFs")
print("   Table structure extraction: enabled\n")

converter = create_configured_converter()
conv_result = converter.convert(str(pdf_path))

print(f"\n✅ PDF converted successfully!")
print(f"\n📊 Document structure:")
print(f"   Tables: {len(conv_result.document.tables)}")
print(f"   Pages: {conv_result.document.export_to_markdown().count('---')}")  # Rough estimate

# Show first table sample
if len(conv_result.document.tables) > 0:
    print(f"\n📋 First table sample (3 rows):")
    for row_idx, row in enumerate(conv_result.document.tables[0].data.grid[:3]):
        if row_idx == 0:
            print(f"   Header: {row[0].text[:60] if hasattr(row[0], 'text') else 'N/A'}...")
        else:
            cell_text = row[0].text[:60] if hasattr(row[0], 'text') else 'N/A'
            print(f"   Row {row_idx}: {cell_text}...")

print(f"\n💡 Key insight: Tables contain requirements 1.1.1, 1.2.3, etc.")
print(f"   But level 1 (1, 2, 3) and level 2 (1.1, 1.2) are in markdown!")

### Cell 2.2: The Hybrid Extraction Approach

**Why not use just one extraction method?**

PCI-DSS (like many compliance documents) has **inconsistent structure**:

| Requirement Level | Format | Extraction Method |
|-------------------|--------|-------------------|
| Level 1 (1, 2, 3...) | Plain text OR `##` headings | **Markdown parsing** |
| Level 2 (1.1, 1.2...) | Markdown list items `- X.Y` | **Markdown parsing** |
| Level 3+ (1.2.3, 1.2.3.4) | Table rows | **Structured table API** |

**🎯 Lesson**: Don't assume one extraction method works for all content!

**The hybrid approach**:
1. Export markdown from Docling (one-time operation)
2. Extract level 1 & 2 using regex on markdown
3. Extract level 3+ using `table.data.grid` API
4. Merge all results

This is **2-3x more accurate** than pure markdown parsing or pure table extraction.

In [ ]:
# Cell 2.3: Extract Level 1 & 2 from Markdown

# Export markdown (reuse conversion result from Cell 2.1)
markdown = conv_result.document.export_to_markdown()

print(f"📄 Exported markdown: {len(markdown)} characters\n")

# Extract Level 1 requirements (principal)
pattern_level_1 = r'^#{0,2}\s*Requirement\s+(\d+):\s*(.+)$'
level_1_reqs = {}

for line in markdown.split('\n'):
    match = re.match(pattern_level_1, line.strip())
    if match:
        req_id = match.group(1)
        title = match.group(2).strip()
        
        if req_id.isdigit() and 1 <= int(req_id) <= 12:
            level_1_reqs[req_id] = {
                'id': req_id,
                'level': 1,
                'parentId': None,
                'title': title,
                'statement': title
            }

print(f"✅ Found {len(level_1_reqs)} level 1 (principal) requirements:\n")
for req_id, req in list(level_1_reqs.items())[:3]:
    print(f"   {req_id}: {req['title'][:60]}...")

# Extract Level 2 requirements (sub-requirements)
pattern_level_2 = r'^-\s+(\d+\.\d+)\s+(.+)$'
level_2_reqs = {}

for line in markdown.split('\n'):
    match = re.match(pattern_level_2, line.strip())
    if match:
        req_id = match.group(1)
        title = match.group(2).strip()
        
        if req_id.count('.') == 1:  # X.Y format
            parent_id = req_id.split('.')[0]
            level_2_reqs[req_id] = {
                'id': req_id,
                'level': 2,
                'parentId': parent_id,
                'title': title,
                'statement': title
            }

print(f"\n✅ Found {len(level_2_reqs)} level 2 (sub-requirement) requirements:\n")
for req_id, req in list(level_2_reqs.items())[:3]:
    print(f"   {req_id}: {req['title'][:60]}...")

print(f"\n📊 Progress: {len(level_1_reqs) + len(level_2_reqs)} requirements extracted from markdown")

In [ ]:
# Cell 2.4: Extract Level 3+ from Tables (Structured API)

def extract_from_tables(document):
    """Extract detailed requirements from tables using Docling's structured API"""
    requirements = {}
    
    for table_ix, table in enumerate(document.tables):
        for row_idx, row in enumerate(table.data.grid):
            if row_idx == 0:  # Skip header row
                continue
            
            # Get cell text safely
            cell_0_text = row[0].text if len(row) > 0 and hasattr(row[0], 'text') else ''
            cell_1_text = row[1].text if len(row) > 1 and hasattr(row[1], 'text') else ''
            
            # Look for "Defined Approach Requirements X.Y.Z"
            if 'Defined Approach Requirements' in cell_0_text:
                # Extract requirement ID
                req_match = re.search(r'(\d+(?:\.\d+){2,})', cell_0_text)
                if req_match:
                    req_id = req_match.group(1)
                    level = len(req_id.split('.'))
                    
                    # Clean statement (remove prefix)
                    statement = re.sub(
                        r'^Defined Approach Requirements\s+\d+(?:\.\d+)+\s*',
                        '',
                        cell_0_text
                    ).strip()
                    
                    # Extract objective from column 2
                    objective = re.sub(
                        r'^Customized Approach Objective\s*',
                        '',
                        cell_1_text
                    ).strip()
                    
                    # Calculate parent ID
                    parent_id = '.'.join(req_id.split('.')[:-1]) if level > 1 else None
                    
                    requirements[req_id] = {
                        'id': req_id,
                        'level': level,
                        'parentId': parent_id,
                        'title': statement[:100] + '...' if len(statement) > 100 else statement,
                        'statement': statement,
                        'customizedApproach': {'objective': objective}
                    }
    
    return requirements

# Extract from tables
detailed_reqs = extract_from_tables(conv_result.document)

print(f"✅ Found {len(detailed_reqs)} level 3+ (detailed) requirements from tables\n")
print(f"Sample requirement IDs:")
for req_id in list(detailed_reqs.keys())[:5]:
    print(f"   {req_id}: {detailed_reqs[req_id]['title'][:60]}...")

# Combine all extractions
all_requirements = {**level_1_reqs, **level_2_reqs, **detailed_reqs}

print(f"\n📊 Total extracted: {len(all_requirements)} requirements")
print(f"   Level 1: {len(level_1_reqs)}")
print(f"   Level 2: {len(level_2_reqs)}")
print(f"   Level 3+: {len(detailed_reqs)}")
print(f"\n🎯 Expected for PCI-DSS v4.0.1: ~267 requirements")

if len(all_requirements) >= 260:
    print(f"✅ Extraction successful!")
else:
    print(f"⚠️  Lower than expected - review extraction logic")

## Part 3: MCP Server Architecture - Building Fast, Intelligent Tools

Now that we've extracted requirements, let's understand how to build an MCP server that AI assistants can use.

###  What is a Model Context Protocol (MCP) Server?

MCP servers provide **tools** that AI assistants like Claude can call to:
- 🔍 Search structured data (e.g., "find firewall requirements")
- 📄 Retrieve specific records (e.g., "get requirement 1.2.3")  
- 🗺️ Map between frameworks (e.g., "map PCI-DSS to NIST")
- ✅ Validate compliance (e.g., "check if policy meets 12.8.1")

**This PCI-DSS server provides 15+ tools** for working with requirements.

### Learning Objectives:
1. Build a **fast index** using Maps (O(1) lookups vs O(n) linear search)
2. Understand **tool definitions** (schema + execute pattern)
3. Implement **search with scoring** (TF-IDF)
4. Design **caching strategies** for performance

In [ ]:
# Cell 3.1: Building a Fast Index with Maps

from typing import Dict, Any, List

class RequirementIndex:
    """
    Fast index for requirement lookups.
    
    Uses Python dict (HashMap) for O(1) lookups instead of O(n) linear search.
    """
    
    def __init__(self):
        self.requirements: Dict[str, Dict[str, Any]] = {}  # ID -> requirement (O(1))
        self.by_level: Dict[int, List[str]] = {}  # Level -> list of IDs
        
    def add(self, requirement: Dict[str, Any]):
        """Add a requirement to the index"""
        req_id = requirement['id']
        level = requirement['level']
        
        # O(1) insertion
        self.requirements[req_id] = requirement
        
        # Group by level for filtering
        if level not in self.by_level:
            self.by_level[level] = []
        self.by_level[level].append(req_id)
    
    def get(self, req_id: str) -> Dict[str, Any]:
        """O(1) lookup by ID"""
        return self.requirements.get(req_id)
    
    def get_by_level(self, level: int) -> List[Dict[str, Any]]:
        """Get all requirements at a specific level"""
        req_ids = self.by_level.get(level, [])
        return [self.requirements[id] for id in req_ids]
    
    def get_children(self, parent_id: str) -> List[Dict[str, Any]]:
        """Get all children of a requirement"""
        children = []
        for req in self.requirements.values():
            if req.get('parentId') == parent_id:
                children.append(req)
        return sorted(children, key=lambda r: r['id'])

# Build the index from extracted requirements
print("🔨 Building index from extracted requirements...\n")
index = RequirementIndex()

for req in all_requirements.values():
    index.add(req)

print(f"✅ Indexed {len(index.requirements)} requirements")
print(f"\n📊 Distribution by level:")
for level in sorted(index.by_level.keys()):
    count = len(index.by_level[level])
    print(f"   Level {level}: {count} requirements")

# Test O(1) lookup
print(f"\n🔍 Test: Get requirement 1.2.3")
req = index.get("1.2.3")
if req:
    print(f"   Found: {req['title'][:60]}...")
else:
    print(f"   Not found")

# Test children lookup
print(f"\n👨‍👩‍👧‍👦 Test: Get children of requirement 1.2")
children = index.get_children("1.2")
print(f"   Found {len(children)} children:")
for child in children[:3]:
    print(f"   - {child['id']}: {child['title'][:50]}...")

In [ ]:
# Cell 3.2: Why Maps Beat Linear Search - Benchmark

import time

def linear_search(requirements_list: List[Dict], target_id: str):
    """O(n) linear search - slow!"""
    for req in requirements_list:
        if req['id'] == target_id:
            return req
    return None

# Convert dict to list for linear search
requirements_list = list(index.requirements.values())

# Benchmark: 1000 lookups
iterations = 1000
target_id = "12.6.3"  # Last requirement - worst case for linear

print(f"⏱️  Benchmarking {iterations} lookups for requirement '{target_id}':\n")

# Test 1: Linear Search (O(n))
start = time.time()
for _ in range(iterations):
    linear_search(requirements_list, target_id)
linear_time = time.time() - start

print(f"Linear Search (O(n)):")
print(f"   Time: {linear_time*1000:.2f}ms for {iterations} lookups")
print(f"   Per lookup: {linear_time*1000/iterations:.4f}ms")

# Test 2: Map Lookup (O(1))
start = time.time()
for _ in range(iterations):
    index.get(target_id)
map_time = time.time() - start

print(f"\nMap Lookup (O(1)):")
print(f"   Time: {map_time*1000:.2f}ms for {iterations} lookups")
print(f"   Per lookup: {map_time*1000/iterations:.4f}ms")

# Comparison
speedup = linear_time / map_time if map_time > 0 else 0
print(f"\n🚀 Speedup: {speedup:.1f}x faster with Maps!")
print(f"\n💡 Key insight: With 267 requirements, Map lookups are ~{len(requirements_list)//2}x faster on average")
print(f"   For 10,000+ documents (like all NIST controls), the difference is dramatic!")

# Show scaling
print(f"\n📈 Scaling comparison:")
for n in [100, 1000, 10000]:
    linear_cost = n // 2  # Average case
    map_cost = 1
    print(f"   {n:,} docs: Linear = ~{linear_cost:,} ops, Map = {map_cost} op ({linear_cost//map_cost}x faster)")

## Part 4: MCP Tool Patterns - Building Your Own Tools

Every MCP tool follows the same pattern: **Schema → Execute → Return**

### Real TypeScript Example (from the PCI-DSS MCP Server)

```typescript
import { z } from "zod";

// 1. Define Zod schema (automatic validation)
const GetRequirementSchema = z.object({
  id: z.string(),
  includeChildren: z.boolean().default(false),
});

// 2. Infer TypeScript type from schema
type GetRequirementInput = z.infer<typeof GetRequirementSchema>;

// 3. Define result interface
interface GetRequirementResult {
  requirement: PciDssRequirement;
  children?: PciDssRequirement[];
}

// 4. Export tool definition
export const getRequirementTool = {
  name: "get_requirement",
  description: "Get full details for a specific PCI-DSS requirement by ID",
  schema: GetRequirementSchema,
  execute: async (input: GetRequirementInput) => {
    const requirement = getRequirement(input.id);

    if (!requirement) {
      throw createError({
        code: "NOT_FOUND",
        message: `Requirement not found: ${input.id}`,
        hint: "Use list_requirements to see available IDs",
      });
    }

    const result = { requirement };

    if (input.includeChildren) {
      result.children = getChildrenRequirements(input.id);
    }

    return result;
  },
};
```

**Why Zod?**
- ✅ Automatic input validation
- ✅ Type-safe (schema generates TypeScript types)
- ✅ Clear error messages when validation fails
- ✅ Self-documenting code

Now let's simulate this pattern in Python!

In [ ]:
# Cell 4.1: Python Simulation of MCP Tool Pattern

from typing import Optional
from dataclasses import dataclass

@dataclass
class ToolInput:
    """Simulates Zod schema validation"""
    id: str
    include_children: bool = False

@dataclass
class ToolResult:
    """Tool output structure"""
    requirement: Dict[str, Any]
    children: Optional[List[Dict[str, Any]]] = None

def get_requirement_tool(input: ToolInput, index: RequirementIndex) -> ToolResult:
    """
    Simulates the MCP get_requirement tool.
    
    Pattern:
    1. Validate input (done by dataclass)
    2. Execute business logic
    3. Handle errors with helpful messages
    4. Return structured result
    """
    requirement = index.get(input.id)

    if not requirement:
        raise ValueError(f"Requirement not found: {input.id}\nHint: Use list_requirements to see available IDs")

    result = ToolResult(requirement=requirement)

    if input.include_children:
        # Find children (requirements with this ID as parent)
        result.children = index.get_children(input.id)

    return result

# Test the tool
print("🔧 Testing get_requirement_tool:\n")

# Test 1: Get requirement without children
input1 = ToolInput(id="1.2.3", include_children=False)
result1 = get_requirement_tool(input1, index)

print(f"Test 1: Get 1.2.3 (no children)")
print(f"   Title: {result1.requirement['title'][:60]}...")
print(f"   Children: {result1.children}")

# Test 2: Get requirement with children
input2 = ToolInput(id="1.2", include_children=True)
result2 = get_requirement_tool(input2, index)

print(f"\nTest 2: Get 1.2 (with children)")
print(f"   Title: {result2.requirement['title'][:60]}...")
print(f"   Children: {len(result2.children) if result2.children else 0}")
if result2.children:
    for child in result2.children[:3]:
        print(f"   - {child['id']}: {child['title'][:50]}...")

print(f"\n✅ Tool pattern works! This is how MCP servers handle requests.")

### 🎯 Exercise: Build Your Own Tool

**Task**: Create a `search_by_level_tool` that:
1. Takes a `level` parameter (1, 2, 3, etc.)
2. Returns all requirements at that level
3. Includes a count of results

**Bonus**: Add a `limit` parameter with a default of 10

Try implementing this before running the solution cell below!

In [ ]:
# Cell 4.2: Solution - search_by_level_tool

@dataclass
class SearchByLevelInput:
    level: int
    limit: int = 10

@dataclass
class SearchByLevelResult:
    level: int
    total_count: int
    returned_count: int
    requirements: List[Dict[str, Any]]

def search_by_level_tool(input: SearchByLevelInput, index: RequirementIndex) -> SearchByLevelResult:
    """Search for all requirements at a specific level"""
    
    # Get all requirements at level
    all_reqs_at_level = index.get_by_level(input.level)
    
    # Apply limit
    limited_reqs = all_reqs_at_level[:input.limit]
    
    return SearchByLevelResult(
        level=input.level,
        total_count=len(all_reqs_at_level),
        returned_count=len(limited_reqs),
        requirements=limited_reqs
    )

# Test it
print("🔍 Testing search_by_level_tool:\n")

for level in [1, 2, 3]:
    input = SearchByLevelInput(level=level, limit=5)
    result = search_by_level_tool(input, index)
    
    print(f"Level {level}:")
    print(f"   Total: {result.total_count}, Showing: {result.returned_count}")
    for req in result.requirements[:2]:
        print(f"   - {req['id']}: {req['title'][:50]}...")
    print()

print("✅ Your custom tool works! This is how you'd add new tools to the MCP server.")

## Part 5: Search & Performance 🚀

Now that you understand basic tools, let's explore advanced features that make your MCP server production-ready:

1. **Full-text search** with TF-IDF scoring (like the real server uses)
2. **Caching strategies** for instant startup times

These patterns are crucial for building responsive, scalable MCP servers.

In [ ]:
# Cell 5.1: Full-Text Search with TF-IDF Scoring
from collections import Counter
import math

class SimpleSearchIndex:
    """
    TF-IDF search implementation for requirement text.
    
    TF-IDF (Term Frequency-Inverse Document Frequency):
    - TF: How often a word appears in a document
    - IDF: How rare the word is across all documents
    - Score: TF × IDF (rewards documents with rare query words)
    """
    def __init__(self, requirements: List[Dict[str, Any]]):
        self.requirements = {req['id']: req for req in requirements}
        self.inverted_index: Dict[str, List[Tuple[str, int]]] = {}
        self.doc_count = len(requirements)
        self._build_index()
    
    def _build_index(self):
        """Build inverted index: word → [(doc_id, term_frequency), ...]"""
        for req in self.requirements.values():
            # Combine searchable fields
            text = f"{req['id']} {req['title']} {req.get('statement', '')}"
            words = text.lower().split()
            
            # Count word frequencies in this document
            word_counts = Counter(words)
            
            for word, count in word_counts.items():
                if word not in self.inverted_index:
                    self.inverted_index[word] = []
                self.inverted_index[word].append((req['id'], count))
    
    def search(self, query: str, limit: int = 10) -> List[Tuple[str, float]]:
        """Search with TF-IDF scoring"""
        query_words = query.lower().split()
        scores: Dict[str, float] = {}
        
        for word in query_words:
            if word not in self.inverted_index:
                continue
            
            # IDF = log(total_docs / docs_containing_word)
            docs_with_word = len(self.inverted_index[word])
            idf = math.log(self.doc_count / docs_with_word)
            
            for doc_id, term_freq in self.inverted_index[word]:
                if doc_id not in scores:
                    scores[doc_id] = 0.0
                
                # TF-IDF score
                scores[doc_id] += term_freq * idf
        
        # Sort by score (highest first)
        sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_results[:limit]

# Build search index from our extracted requirements
print("Building search index...")
search_index = SimpleSearchIndex(list(all_requirements.values()))
print(f"✓ Indexed {len(search_index.requirements)} requirements")
print(f"  Vocabulary size: {len(search_index.inverted_index)} unique words\n")

# Test search
query = "firewall network security controls"
print(f"Searching for: '{query}'")
print("-" * 60)

results = search_index.search(query, limit=5)
print(f"\nTop {len(results)} results:\n")

for req_id, score in results:
    req = search_index.requirements[req_id]
    print(f"[{req_id}] Score: {score:.2f}")
    print(f"  {req['title'][:80]}...")
    print()

# The real MCP server uses Lunr.js (JavaScript library)
# This Python implementation shows the same concept!

In [ ]:
# Cell 5.2: Caching Strategy for Instant Startup
import json
import hashlib
from pathlib import Path
import time

class CachedIndex:
    """
    Cache extracted requirements to avoid re-parsing PDFs.
    
    Strategy:
    - Cache key: SHA256 hash of PDF (detects changes)
    - Cache location: ~/.cache/pcidss-docs/
    - Invalidation: Automatic when PDF changes
    """
    def __init__(self, cache_dir: Path):
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
    
    def get_cache_key(self, pdf_path: Path) -> str:
        """Generate cache key from PDF hash"""
        pdf_hash = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
        return f"index-{pdf_hash[:16]}.json"
    
    def load(self, pdf_path: Path) -> Optional[Dict]:
        """Load index from cache if valid"""
        cache_file = self.cache_dir / self.get_cache_key(pdf_path)
        
        if cache_file.exists():
            with open(cache_file) as f:
                cached = json.load(f)
            print(f"✓ Loaded from cache: {cache_file.name}")
            return cached
        
        return None
    
    def save(self, pdf_path: Path, index_data: Dict):
        """Save index to cache"""
        cache_file = self.cache_dir / self.get_cache_key(pdf_path)
        
        with open(cache_file, 'w') as f:
            json.dump(index_data, f, indent=2)
        
        file_size = cache_file.stat().st_size / 1024  # KB
        print(f"✓ Saved to cache: {cache_file.name} ({file_size:.1f} KB)")

# Demonstrate caching impact
cache = CachedIndex(Path(".cache"))

# Simulate expensive operation (PDF extraction)
def build_index_slow():
    """Simulate slow PDF extraction"""
    time.sleep(0.5)  # Pretend we're parsing a PDF
    return {
        'requirements': list(all_requirements.values()),
        'metadata': {
            'version': '4.0.1',
            'cached_at': datetime.now().isoformat()
        }
    }

print("Performance Comparison")
print("=" * 60)

# First run: No cache (slow)
print("\n1️⃣  First run (no cache):")
start = time.time()
index_data = build_index_slow()
cache.save(pdf_path, index_data)
first_run_time = (time.time() - start) * 1000  # ms

# Second run: With cache (fast)
print("\n2️⃣  Second run (with cache):")
start = time.time()
cached_data = cache.load(pdf_path)
second_run_time = (time.time() - start) * 1000  # ms

print(f"\n📊 Results:")
print(f"  No cache:   {first_run_time:.0f}ms")
print(f"  With cache: {second_run_time:.0f}ms")
print(f"  Speedup:    {first_run_time/second_run_time:.1f}x faster")

print(f"\n💡 Real server startup:")
print(f"  First run:  ~230 seconds (PDF extraction)")
print(f"  Cached:     ~45ms (just loading JSON)")
print(f"  Speedup:    ~5000x faster!")

## Part 6: Testing & Validation ✅

Before deploying your MCP server, you need to ensure data quality and functionality. This section covers:

1. **Validation tests** - Verify extracted data meets quality standards
2. **Integration tests** - Test the complete workflow end-to-end

These patterns help catch extraction errors early and ensure your server behaves correctly.

In [ ]:
# Cell 6.1: Validation Tests
def validate_requirements(requirements: Dict[str, Dict[str, Any]]) -> List[str]:
    """
    Validate extracted requirements meet quality standards.
    
    Checks:
    1. Expected count (~267 for PCI-DSS v4.0.1)
    2. Level distribution (12 L1, 63 L2, etc.)
    3. Hierarchy validity (all parents exist)
    4. Required fields present
    
    Returns: List of error messages (empty if valid)
    """
    errors = []
    
    # Check 1: Expected count
    if len(requirements) < 260:
        errors.append(f"⚠️  Only {len(requirements)} requirements (expected ~267)")
    
    # Check 2: Level distribution
    expected_levels = {
        1: 12,   # Principal requirements
        2: 63,   # Sub-requirements
        3: 153,  # Detailed requirements (most common)
    }
    
    actual_levels = {}
    for req in requirements.values():
        level = req['level']
        actual_levels[level] = actual_levels.get(level, 0) + 1
    
    for level, expected_count in expected_levels.items():
        actual_count = actual_levels.get(level, 0)
        if actual_count < expected_count * 0.9:  # Allow 10% variance
            errors.append(
                f"⚠️  Level {level}: found {actual_count}, expected ~{expected_count}"
            )
    
    # Check 3: Hierarchy validity
    for req_id, req in requirements.items():
        if req['level'] > 1:
            parent_id = req.get('parentId')
            if not parent_id:
                errors.append(f"❌ {req_id}: missing parentId")
            elif parent_id not in requirements:
                errors.append(f"❌ {req_id}: parent '{parent_id}' not found")
    
    # Check 4: Required fields
    required_fields = ['id', 'level', 'title', 'statement']
    for req_id, req in requirements.items():
        for field in required_fields:
            if field not in req or not req[field]:
                errors.append(f"❌ {req_id}: missing required field '{field}'")
    
    return errors

# Run validation
print("Running validation tests...")
print("=" * 60)

errors = validate_requirements(all_requirements)

if errors:
    print(f"❌ Found {len(errors)} validation errors:\n")
    for error in errors[:10]:  # Show first 10
        print(f"  {error}")
    if len(errors) > 10:
        print(f"\n  ... and {len(errors) - 10} more errors")
else:
    print("✅ All validation checks passed!")
    print(f"\n📊 Statistics:")
    print(f"  Total requirements: {len(all_requirements)}")
    
    # Level breakdown
    levels = {}
    for req in all_requirements.values():
        level = req['level']
        levels[level] = levels.get(level, 0) + 1
    
    for level in sorted(levels.keys()):
        print(f"  Level {level}: {levels[level]} requirements")
    
    # Hierarchy check
    orphans = sum(1 for req in all_requirements.values() 
                  if req['level'] > 1 and not req.get('parentId'))
    print(f"  Orphaned requirements: {orphans}")

In [ ]:
# Cell 6.2: Integration Test
def integration_test():
    """
    Test complete workflow: Extract → Index → Search → Validate
    
    This simulates what happens when your MCP server starts up.
    """
    print("🧪 Integration Test")
    print("=" * 60)
    
    # Step 1: Extract (we've already done this, but verify)
    print("\n1️⃣  PDF Extraction")
    reqs = all_requirements
    assert len(reqs) > 0, "❌ No requirements extracted"
    print(f"   ✓ Extracted {len(reqs)} requirements")
    
    # Step 2: Build index
    print("\n2️⃣  Index Building")
    idx = RequirementIndex()
    for req in reqs.values():
        idx.add(req)
    assert len(idx.requirements) == len(reqs), "❌ Index size mismatch"
    print(f"   ✓ Indexed {len(idx.requirements)} requirements")
    
    # Step 3: Test tools
    print("\n3️⃣  Tool Testing")
    
    # Test get_requirement
    test_req = idx.get("1.2.3")
    assert test_req is not None, "❌ Failed to get requirement 1.2.3"
    assert test_req['id'] == "1.2.3", "❌ Wrong requirement returned"
    print(f"   ✓ get_requirement works")
    
    # Test level filtering
    level_1_reqs = idx.get_by_level(1)
    assert len(level_1_reqs) == 12, f"❌ Expected 12 L1 reqs, got {len(level_1_reqs)}"
    print(f"   ✓ get_by_level works ({len(level_1_reqs)} level 1 reqs)")
    
    # Test search
    search_idx = SimpleSearchIndex(list(reqs.values()))
    results = search_idx.search("firewall", limit=5)
    assert len(results) > 0, "❌ Search returned no results"
    print(f"   ✓ search works ({len(results)} results)")
    
    # Step 4: Validate
    print("\n4️⃣  Data Validation")
    errors = validate_requirements(reqs)
    if errors:
        print(f"   ⚠️  Found {len(errors)} validation warnings")
        for error in errors[:3]:
            print(f"      {error}")
    else:
        print(f"   ✓ Validation passed")
    
    # Step 5: Verify hierarchy
    print("\n5️⃣  Hierarchy Testing")
    # Test parent-child relationship
    parent = idx.get("1.2")
    children = [req for req in idx.requirements.values() 
                if req.get('parentId') == "1.2"]
    assert len(children) > 0, "❌ No children found for 1.2"
    print(f"   ✓ Hierarchy works (1.2 has {len(children)} children)")
    
    print("\n" + "=" * 60)
    print("✅ Integration test PASSED!")
    print("\n💡 Your extraction pipeline is working correctly.")
    print("   Ready to build the MCP server!")

# Run integration test
try:
    integration_test()
except AssertionError as e:
    print(f"\n❌ Integration test FAILED: {e}")
except Exception as e:
    print(f"\n❌ Unexpected error: {e}")
    import traceback
    traceback.print_exc()

## Part 7: Real Integration & Adaptation 🚀

You've learned the complete pipeline! Now let's:

1. **Save your data** in the MCP server format
2. **Learn how to adapt** this pattern for other compliance docs (FedRAMP, SOC2, HIPAA, etc.)

This is where you take these patterns and build your own MCP servers.

In [ ]:
# Cell 7.1: Save Converted Data in MCP Server Format
from datetime import datetime

# Prepare output in MCP server format
output = {
    'info': {
        'version': '4.0.1',
        'published': '2024-03-31',
        'effectiveDate': '2024-03-31',
        'documentType': 'standard',
        'convertedAt': datetime.now().isoformat(),
        'convertedBy': f'docling {docling_version}',
        'pdfHash': hashlib.sha256(pdf_path.read_bytes()).hexdigest(),
    },
    'requirements': sorted(
        all_requirements.values(),
        key=lambda r: [int(x) if x.isdigit() else 999 for x in r['id'].split('.')]
    ),
    'appendices': [],  # Could be extracted from PDF sections
    'glossary': []     # Could be extracted from PDF glossary
}

# Save to output directory
output_file = output_dir / 'requirements.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

file_size = output_file.stat().st_size / 1024  # KB

print("✅ Saved MCP Server Data")
print("=" * 60)
print(f"\n📄 File: {output_file}")
print(f"📦 Size: {file_size:.1f} KB")
print(f"📊 Requirements: {len(output['requirements'])}")
print(f"🔐 PDF Hash: {output['info']['pdfHash'][:16]}...")

print("\n💡 Next Steps:")
print("  1. Copy this file to your MCP server: data/converted/requirements.json")
print("  2. Update TypeScript types in src/types.ts (if needed)")
print("  3. Run npm install && npm run build")
print("  4. Add to Claude Desktop config")
print("\n  Your MCP server is ready to use! 🎉")

## 🎓 You've Mastered MCP Server Building!

### Key Concepts You've Learned

1. **PDF Extraction with Docling**
   - Hybrid approach (markdown + structured tables)
   - Handle inconsistent document structure
   - Validate extraction quality

2. **Indexing for Performance**
   - Map-based O(1) lookups
   - Level-based filtering
   - Parent-child relationships

3. **MCP Tool Patterns**
   - Zod schemas for validation
   - Error handling with helpful hints
   - Type-safe tool definitions

4. **Search & Caching**
   - TF-IDF scoring for relevance
   - Inverted index for fast lookup
   - Cache invalidation strategies

---

### 📚 Adapting This for Other Compliance Docs

The same patterns work for **any** hierarchical compliance document. Here's how:

#### **FedRAMP Controls (NIST 800-53)**

```python
def extract_fedramp(pdf_path):
    """
    Structure: Control Families → Controls → Enhancements
    
    Example hierarchy:
    - AC (Access Control) - Level 1
      - AC-1 (Policy and Procedures) - Level 2
        - AC-1(1) (Enhancement) - Level 3
    """
    # Level 1: Control families (extract from headings)
    pattern_family = r'^([A-Z]{2,3})\s*-\s*(.+)$'
    
    # Level 2: Controls (extract from tables)
    pattern_control = r'^([A-Z]{2,3}-\d+)\s+(.+)$'
    
    # Level 3: Enhancements (extract from nested tables)
    pattern_enhancement = r'^([A-Z]{2,3}-\d+\(\d+\))\s+(.+)$'
    
    # Hybrid extraction (same approach!)
    families = extract_from_markdown(markdown, pattern_family)
    controls = extract_from_tables(document, pattern_control)
    enhancements = extract_from_tables(document, pattern_enhancement)
    
    return merge_requirements(families, controls, enhancements)
```

**Key differences from PCI-DSS:**
- IDs use letters (AC-1 vs 1.2.3)
- Enhancements use parentheses AC-1(1)
- More emphasis on control objectives

---

#### **SOC 2 Trust Service Criteria**

```python
def extract_soc2(pdf_path):
    """
    Structure: Categories → Criteria → Points of Focus
    
    Example hierarchy:
    - CC (Common Criteria) - Level 1
      - CC1.1 (Criteria) - Level 2
        - Point of Focus 1 - Level 3
    """
    # Level 1: Trust service categories (CC, A, PI, C)
    pattern_category = r'^([A-Z]{1,2})\s*-\s*(.+)$'
    
    # Level 2: Criteria (CC1.1, CC1.2)
    pattern_criteria = r'^([A-Z]{1,2}\d+\.\d+)\s+(.+)$'
    
    # Level 3: Points of focus (bullets under criteria)
    # Extract from markdown lists
    
    return extract_and_merge(...)
```

**Key differences from PCI-DSS:**
- Criteria IDs use dot notation (CC1.1)
- Points of focus often unnumbered
- Focus on controls vs requirements

---

#### **HIPAA Security Rule**

```python
def extract_hipaa(pdf_path):
    """
    Structure: Safeguards → Standards → Implementation Specs
    
    Example hierarchy:
    - Administrative Safeguards - Level 1
      - 164.308(a)(1) Security Management - Level 2
        - (i) Risk Analysis (Required) - Level 3
    """
    # Level 1: Safeguard categories
    pattern_safeguard = r'^(Administrative|Physical|Technical)\s+Safeguards'
    
    # Level 2: Standards (CFR references)
    pattern_standard = r'^§?\s*164\.(\d+)\(([a-z])\)\((\d+)\)\s+(.+)$'
    
    # Level 3: Implementation specs
    # Often in sub-sections with (i), (ii), (iii)
    
    return extract_and_merge(...)
```

**Key differences from PCI-DSS:**
- CFR (Code of Federal Regulations) numbering
- Required vs Addressable distinction
- Less hierarchical (flatter structure)

---

### 🛠️ Building Your MCP Server

**Step 1: Clone this repo as template**
```bash
git clone https://github.com/ethanolivertroy/pcidss-docs-mcp my-mcp-server
cd my-mcp-server
```

**Step 2: Replace PDF extraction**
- Adapt extraction patterns for your document
- Test with `python scripts/docling_convert.py your.pdf data/converted`

**Step 3: Update TypeScript types** (`src/types.ts`)
```typescript
export interface YourRequirement {
  id: string;
  level: number;
  parentId: string | null;
  title: string;
  statement: string;
  // Add your custom fields here
}
```

**Step 4: Adapt tool descriptions** (`src/tools/*.ts`)
- Update descriptions for your domain
- Add domain-specific tools (e.g., map_to_nist, find_evidence_requirements)

**Step 5: Test and deploy**
```bash
npm install
npm run build
npm test
```

---

### 📖 Resources

- **MCP SDK**: https://github.com/modelcontextprotocol/typescript-sdk
- **Docling docs**: https://docling.readthedocs.io/
- **This repo**: https://github.com/ethanolivertroy/pcidss-docs-mcp
- **Claude Desktop**: https://claude.ai/download

---

### ⚖️ License Reminder

**Always respect document licenses!**

This notebook teaches you to build **tools**, not to redistribute content:
- ✅ Build extraction tools for internal use
- ✅ Create MCP servers for your organization
- ✅ Automate compliance workflows
- ❌ Don't redistribute extracted PCI-DSS data
- ❌ Don't share converted requirements publicly
- ❌ Don't violate document copyright

---

### 🎯 What's Next?

You now have the skills to:
1. Extract structured data from complex PDFs
2. Build performant MCP servers
3. Create custom tools for AI assistants
4. Adapt this pattern to any compliance framework

**Go build something!** 🚀

Questions? Open an issue on GitHub (without sharing converted data).

## Step 4: Run Conversion

This cell runs the conversion script that:
1. Parses the PDF using Docling
2. Extracts hierarchical requirements (1, 1.1, 1.1.1, etc.)
3. Identifies testing procedures (1.a, 1.b, etc.)
4. Extracts customized approach objectives
5. Generates structured JSON and searchable Markdown

**Note:** This may take 1-2 minutes depending on PDF size and complexity.

In [ ]:
import subprocess
import sys

# Get path to conversion script
if IN_COLAB:
    # Clone the repo to get the script
    if not Path("pcidss-docs-mcp").exists():
        print("📦 Cloning repository...")
        !git clone -q https://github.com/ethanolivertroy/pcidss-docs-mcp.git
    script_path = "pcidss-docs-mcp/scripts/docling_convert.py"
else:
    script_path = "../scripts/docling_convert.py"

print("🔄 Converting PDF to JSON/Markdown...\n")
print("This may take 1-2 minutes. Please wait...\n")

result = subprocess.run([
    sys.executable,
    script_path,
    str(pdf_path),
    str(output_dir)
], capture_output=True, text=True)

print(result.stdout)

if result.returncode != 0:
    print("❌ Conversion failed:")
    print(result.stderr)
    raise RuntimeError("PDF conversion failed")

print("\n✅ Conversion completed successfully!")

## Step 5: Validate Output

Let's verify that the conversion was successful and inspect the results.

In [ ]:
import json

requirements_file = output_dir / "requirements.json"
metadata_file = output_dir / "metadata.json"
markdown_file = output_dir / "requirements.md"

# Check all files exist
print("📁 Generated files:\n")
for file in [requirements_file, metadata_file, markdown_file]:
    if file.exists():
        size_kb = file.stat().st_size / 1024
        print(f"  ✅ {file.name}: {size_kb:.1f} KB")
    else:
        print(f"  ❌ {file.name}: MISSING")

# Load and validate requirements.json
with open(requirements_file) as f:
    data = json.load(f)

print(f"\n📊 Conversion Summary:\n")
print(f"  Version: {data['info']['version']}")
print(f"  Published: {data['info']['published']}")
print(f"  Converted: {data['info']['convertedAt']}")
print(f"  Total Requirements: {len(data['requirements'])}")

# Show first requirement as example
first_req = data['requirements'][0]
print(f"\n📋 First Requirement Example:\n")
print(f"  ID: {first_req['id']}")
print(f"  Title: {first_req['title']}")
print(f"  Level: {first_req['level']}")
print(f"  Statement: {first_req['statement'][:100]}...")

print("\n✅ Validation complete! Your PCI-DSS data is ready.")

## Step 6: Download Converted Files (Google Colab Only)

If you're running in Google Colab, download the converted files to your computer.

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    print("📥 Downloading converted files...\n")
    
    for file in [requirements_file, metadata_file, markdown_file]:
        if file.exists():
            print(f"  Downloading {file.name}...")
            files.download(str(file))
    
    print("\n✅ Download complete!")
    print("\nNext steps:")
    print("1. Place the downloaded files in your local repo's data/converted/ directory")
    print("2. Run: npm run build")
    print("3. Run: npm run dev")
else:
    print("💻 Running locally - files are already in the correct location!")
    print("\nNext steps:")
    print("1. cd .. (back to repo root)")
    print("2. npm run build")
    print("3. npm run dev")

## Step 7: Next Steps

### You've Successfully Converted Your PCI-DSS PDF! 🎉

Your converted data is now ready to use with the PCI-DSS MCP server.

### What You Created:

1. **requirements.json** (~371 KB)
   - Structured data with 266 requirements
   - Hierarchical organization (principal → sub → detailed)
   - Testing procedures and customized approach objectives

2. **requirements.md** (~2.2 MB)
   - Full searchable markdown export
   - Useful for grep/search operations

3. **metadata.json** (~260 B)
   - Conversion metadata (timestamp, hash)
   - Used to detect if reconversion needed

### Using the MCP Server:

```bash
# Build the server
npm run build

# Test locally
npm run dev

# Configure Claude Desktop
# Edit: ~/Library/Application Support/Claude/claude_desktop_config.json
# Add:
{
  "mcpServers": {
    "pcidss-docs": {
      "command": "node",
      "args": ["/absolute/path/to/pcidss-docs-mcp/dist/index.js"]
    }
  }
}
```

### Important Reminders:

⚠️ **DO NOT:**
- Share the converted JSON/Markdown files
- Commit them to public repositories
- Redistribute them in any form

✅ **DO:**
- Use them for your internal compliance work
- Keep them in your local/private environment
- Re-convert if you update the PDF

### Troubleshooting:

**Server won't start?**
- Verify `data/converted/requirements.json` exists
- Check `npm run build` completed successfully
- Review error messages for missing dependencies

**Need to reconvert?**
```bash
npm run convert-pdfs
```

**Questions?**
- Check README.md
- Review SETUP.md
- Open an issue on GitHub (without sharing converted data)

---

**License:** This tutorial and conversion tools are MIT licensed.  
**PCI DSS Content:** Copyrighted by PCI Security Standards Council - your license is for internal use only.